# AC-MOT Video Tracker  ·  Colab (self-contained)
Runs the AC-MOT tracking pipeline on a video from Google Drive and writes an
annotated output video (boxes + IDs + live HUD: FPS / SCI / scene / conf) plus
a stats summary. All code is inside this notebook -- nothing to upload.

**Before running:** Runtime -> Change runtime type -> GPU (T4).

In [ ]:
# 1) Mount Drive + install
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics opencv-python

In [ ]:
# 2) The AC-MOT tracking pipeline (run once to define everything)
import cv2, numpy as np, time, json
from collections import deque, defaultdict
from pathlib import Path
from ultralytics import YOLO

class SceneAnalyzer:
    """Scene Complexity Index (SCI in [0,1]) from 5 cheap cues, 7-frame smoothed."""
    def __init__(self, smooth=7):
        self.hist = deque(maxlen=smooth)
    def reset(self):
        self.hist.clear()
    def analyze(self, img, prev_boxes):
        small = cv2.resize(img, (0, 0), fx=0.25, fy=0.25)
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        edge = float(cv2.Canny(gray, 50, 120).mean() / 255.0)
        crowd = min(len(prev_boxes) / 30.0, 1.0)
        if len(prev_boxes):
            areas = (prev_boxes[:,2]-prev_boxes[:,0]) * (prev_boxes[:,3]-prev_boxes[:,1])
            tiny = float(np.mean(areas < 32*32))
        else:
            tiny = 0.0
        raw = 0.30*crowd + 0.20*min(edge/0.14, 1.0) + 0.30*tiny
        night = brightness < 80
        blurry = blur < 180
        if night: raw += 0.10
        if blurry: raw += 0.05
        self.hist.append(float(np.clip(raw, 0, 1)))
        sci = float(np.mean(self.hist))
        if night: scene = "NIGHT"
        elif blurry: scene = "BLUR"
        elif tiny > 0.5: scene = "TINY"
        elif sci >= 0.60: scene = "CROWDED"
        elif sci >= 0.35: scene = "MEDIUM"
        else: scene = "CLEAR"
        return dict(sci=sci, scene=scene)

def calibrate(state):
    sci = state["sci"]; hard = state["scene"] in ("CROWDED","TINY","NIGHT")
    conf = float(np.clip(0.245 - 0.050*sci - 0.012*hard, 0.19, 0.28))
    iou = float(np.clip(0.52 - 0.12*sci, 0.38, 0.55))
    imgsz = 832 if sci > 0.60 else (736 if sci > 0.35 else 640)
    return dict(conf=conf, iou=iou, imgsz=imgsz)

# tuned ByteTrack config (the AC-MOT final tracker)
Path("bytetrack_tuned.yaml").write_text(
    "tracker_type: bytetrack\n"
    "track_high_thresh: 0.18\n"
    "track_low_thresh: 0.04\n"
    "new_track_thresh: 0.20\n"
    "track_buffer: 45\n"
    "match_thresh: 0.86\n"
    "fuse_score: True\n")

def id_color(t):
    rng = (int(t)*9973) % 360
    c = np.uint8([[[int(rng/360*179), 200, 255]]])
    b = cv2.cvtColor(c, cv2.COLOR_HSV2BGR)[0][0]
    return int(b[0]), int(b[1]), int(b[2])

def draw_box(img, box, tid, label):
    x1,y1,x2,y2 = [int(v) for v in box]; col = id_color(tid)
    cv2.rectangle(img,(x1,y1),(x2,y2),col,2)
    tag = "#%d %s" % (int(tid), label)
    (tw,th),_ = cv2.getTextSize(tag,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
    cv2.rectangle(img,(x1,y1-th-6),(x1+tw+4,y1),col,-1)
    cv2.putText(img,tag,(x1+2,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0),1,cv2.LINE_AA)

def draw_hud(img, lines):
    pad,lh,w = 10,22,250; h = pad*2+lh*len(lines)
    ov = img.copy(); cv2.rectangle(ov,(8,8),(8+w,8+h),(10,16,31),-1)
    cv2.addWeighted(ov,0.62,img,0.38,0,img); cv2.rectangle(img,(8,8),(8+w,8+h),(255,214,10),1)
    for i,(t,c) in enumerate(lines):
        cv2.putText(img,t,(16,8+pad+lh*(i+1)-6),cv2.FONT_HERSHEY_SIMPLEX,0.5,c,1,cv2.LINE_AA)

def run_video(video, out_path, model, device=0, classes=None,
              fixed_conf=0.12, fixed_imgsz=1280, adaptive=True, max_frames=None):
    if getattr(model,"predictor",None) is not None:
        model.predictor = None
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened(): raise SystemExit("cannot open " + str(video))
    W=int(cap.get(3)); H=int(cap.get(4)); fps=cap.get(5) or 30; total=int(cap.get(7))
    wr = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W,H))
    sa = SceneAnalyzer(); prev = np.empty((0,4)); params = dict(conf=0.25,iou=0.45,imgsz=640)
    times=[]; det=[]; uids=set(); life=defaultdict(int); names=model.names; idx=0; st={}
    print("video:", Path(video).name, " %dx%d  %d frames" % (W,H,total))
    while True:
        ok, fr = cap.read()
        if not ok: break
        idx += 1
        if max_frames and idx > max_frames: break
        if adaptive and (idx==1 or idx%10==1):
            st = sa.analyze(fr, prev); params = calibrate(st)
        elif not adaptive:
            st = dict(sci=0.0, scene="FIXED"); params = dict(conf=0.25,iou=0.45,imgsz=640)
        if fixed_conf is not None: params["conf"] = fixed_conf
        if fixed_imgsz is not None: params["imgsz"] = fixed_imgsz
        t0 = time.perf_counter()
        res = model.track(source=fr, tracker="bytetrack_tuned.yaml",
                          conf=params["conf"], iou=params["iou"], imgsz=params["imgsz"],
                          persist=True, classes=classes, verbose=False, device=device)
        dt = time.perf_counter()-t0; times.append(dt)
        r = res[0]
        if r.boxes is not None and r.boxes.id is not None:
            ids=r.boxes.id.cpu().numpy().astype(int); boxes=r.boxes.xyxy.cpu().numpy()
            cls=r.boxes.cls.cpu().numpy().astype(int); prev=boxes.copy(); det.append(len(ids))
            for b,ti,c in zip(boxes,ids,cls):
                uids.add(int(ti)); life[int(ti)]+=1
                draw_box(fr,b,ti,names.get(int(c),str(c)))
        else:
            prev=np.empty((0,4)); det.append(0)
        avg = len(times)/sum(times) if times else 0
        draw_hud(fr, [("AC-MOT (ByteTrack tuned)",(0,229,255)),
                      ("FPS %4.1f  (avg %4.1f)" % (1/dt if dt>0 else 0, avg),(57,255,20)),
                      ("SCI %.2f   %s" % (st.get("sci",0), st.get("scene","-")),(255,214,10)),
                      ("conf %.2f   imgsz %d" % (params["conf"], params["imgsz"]),(220,220,220)),
                      ("tracks live %d" % det[-1],(179,136,255))])
        wr.write(fr)
        if idx % 100 == 0:
            print("   %d/%d   avg %4.1f FPS   %d unique IDs" % (idx,total,avg,len(uids)))
    cap.release(); wr.release()
    avg = len(times)/sum(times) if times else 0
    stable = [t for t in life.values() if t > 5]
    summ = dict(video=Path(video).name, frames=idx, avg_fps=round(avg,2),
                unique_ids=len(uids), stable_tracks=len(stable),
                avg_det_per_frame=round(float(np.mean(det)),2) if det else 0,
                output=str(out_path))
    print("wrote", out_path)
    return summ

print("pipeline ready")

### 3) Run on your video
Edit `VIDEO` to your file on Drive. `imgsz 1280 / conf 0.12` is tuned for
high-altitude drone footage; `classes=[2,3,5,7]` = car / motorcycle / bus / truck
(add `0` for people, or set `classes=None` for everything).

In [ ]:
# config -- edit these
DRIVE_DIR = "/content/drive/MyDrive/afv/videos to track"
OUTDIR    = "/content/drive/MyDrive/afv/results"
VIDEO     = DRIVE_DIR + "/Library-3.MP4"

Path(OUTDIR).mkdir(parents=True, exist_ok=True)
model = YOLO("yolov8n.pt")

out = OUTDIR + "/" + Path(VIDEO).stem + "__acmot.mp4"
summary = run_video(VIDEO, out, model, device=0, classes=[2,3,5,7],
                    fixed_conf=0.12, fixed_imgsz=1280)
json.dump(summary, open(OUTDIR + "/" + Path(VIDEO).stem + "__summary.json","w"), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# 4) (optional) download the annotated video to your computer
from google.colab import files
files.download(out)

### (optional) Process every video in the folder at once

In [ ]:
VID_EXT = {".mp4",".MP4",".mov",".MOV",".avi",".mkv"}
vids = sorted(p for p in Path(DRIVE_DIR).iterdir() if p.suffix in VID_EXT)
print("found:", [v.name for v in vids])
all_sum = []
for v in vids:
    o = OUTDIR + "/" + v.stem + "__acmot.mp4"
    s = run_video(str(v), o, model, device=0, classes=[2,3,5,7],
                  fixed_conf=0.12, fixed_imgsz=1280)
    json.dump(s, open(OUTDIR + "/" + v.stem + "__summary.json","w"), indent=2)
    all_sum.append(s)
print("\n=== ALL VIDEOS ===")
for s in all_sum:
    print("%-20s FPS %5s  IDs %4d  stable %4d  det/f %s" % (
        s["video"], s["avg_fps"], s["unique_ids"], s["stable_tracks"], s["avg_det_per_frame"]))